# Seance 7 - NumPy et pandas

**Formation Python 360 - Commission Scientifique nationale - ASEGUIM**

---

## Avant de commencer

Si tu ouvres ce notebook dans Colab : **Fichier > Enregistrer une copie dans
Drive**. Sans cela, tu travailles dans un fichier en lecture seule et rien
ne sera garde.

Ce notebook reprend, dans l'ordre, les cinq fichiers du dossier `reprise/`.
Il se lit de haut en bas : chaque cellule suppose que les precedentes ont
ete executees.

## Ce qu'on fait aujourd'hui

1. Pourquoi NumPy va plus vite qu'une boucle
2. Vectorisation, masque booleen, broadcasting
3. Charger un releve bancaire et l'inspecter
4. Le nettoyer : dates, montants, casse, doublons, trous
5. L'agreger : `groupby`, `agg`, `pivot_table`

## Les trois pieges de la journee

1. `pd.to_datetime(..., format="mixed", dayfirst=True)` inverse des dates
   sans rien dire.
2. `dropna()` sans `subset` supprime beaucoup plus de lignes que prevu.
3. `df[filtre]["colonne"] = valeur` n'a aucun effet : c'est un
   avertissement, pas une erreur.

---

## 0. Verifier les versions

pandas 3.0 est sorti en janvier 2026 et a change plusieurs comportements.
Une version 2.x donnerait des resultats differents sur une partie de ce
notebook.

In [ ]:
import numpy as np
import pandas as pd

print("pandas :", pd.__version__)
print("numpy  :", np.__version__)

# On demande a pandas d'afficher les tableaux en entier.
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

### Charger le relevé

Le fichier `releve_brut.csv` est dans le dossier `data/` du depot. La
fonction ci-dessous le cherche d'abord en local (si tu travailles dans le
depot) et le telecharge depuis GitHub sinon (si tu es dans Colab).

In [ ]:
from pathlib import Path

URL_RELEVE = (
    "https://raw.githubusercontent.com/8sylla/python-360/main/data/releve_brut.csv"
)


def source_releve():
    """Rend le chemin local du releve s'il existe, sinon son URL."""
    for dossier in [Path.cwd(), *Path.cwd().parents]:
        candidat = dossier / "data" / "releve_brut.csv"
        if candidat.exists():
            return candidat
    return URL_RELEVE


print("Source :", source_releve())

---

# 1. NumPy : pourquoi c'est plus rapide

Une liste Python accepte n'importe quoi : des nombres, du texte, d'autres
listes. C'est pratique, et c'est exactement ce qui l'empeche d'etre rapide.

Un tableau NumPy (`ndarray`) ne contient qu'une seule sorte de valeur,
rangee dans un bloc de memoire contigu. La boucle descend alors dans du C
compile au lieu de repasser par l'interpreteur a chaque tour.

In [ ]:
import time

N = 1_000_000
liste = list(range(N))
tableau = np.arange(N)

debut = time.perf_counter()
resultat_liste = [x * 2 for x in liste]
duree_liste = time.perf_counter() - debut

debut = time.perf_counter()
resultat_numpy = tableau * 2
duree_numpy = time.perf_counter() - debut

print(f"liste : {duree_liste * 1000:7.1f} ms")
print(f"numpy : {duree_numpy * 1000:7.1f} ms")
print(f"facteur : x{duree_liste / duree_numpy:.0f}")

Le facteur depend de la machine : entre 20 et 35 sur les postes testes.
Ce qui compte est l'ordre de grandeur, pas la valeur exacte.

Deux raisons, dans cet ordre :

- **Le rangement.** Une liste est un tableau de pointeurs vers des objets
  disperses en memoire. Un `ndarray` est un bloc de nombres bruts.
- **La boucle.** `tableau * 2` ne fait pas un million de tours en Python :
  la boucle est ecrite en C.

Question qui revient toujours : pourquoi ne pas tout faire en NumPy ?
Parce qu'un `ndarray` ne contient qu'un seul type. Des qu'une ligne melange
un texte, une date et un nombre - un releve bancaire, exactement - il faut
pandas.

## 1.1 La vectorisation

On ecrit l'operation sur le tableau entier, et on n'ecrit plus la boucle.

In [ ]:
montants = np.array([12.5, 340.0, 8.9, 127.4])

print("montants * 1.2 :", montants * 1.2)
print("montants > 100 :", montants > 100)
print("filtres        :", montants[montants > 100])

## 1.2 Le masque booleen

La troisieme ligne ci-dessus est le geste central de toute la seance. Elle
contient deux fois le mot `montants` : une fois comme tableau a filtrer, une
fois comme source du masque.

C'est exactement la forme `df[df.montant > 100]` qu'on ecrira toute la
soiree.

In [ ]:
masque = montants > 100

print("les valeurs   :", montants)
print("le masque     :", masque)
print("les gardees   :", montants[masque])
print("leur somme    :", montants[masque].sum())

# Le masque a exactement la longueur du tableau.
print("longueurs     :", len(montants), len(masque))

## 1.3 Le broadcasting

NumPy etire ce qui est unique. Il n'invente jamais ce qui manque.

In [ ]:
# Une seule valeur, etiree sur tout le tableau.
print(montants * 1.2)

# Deux tableaux de MEME longueur, case a case.
remises = np.array([1.0, 10.0, 0.5, 5.0])
print(montants - remises)

# Longueurs incompatibles : NumPy refuse.
try:
    np.arange(3) * np.arange(5)
except ValueError as erreur:
    print("ValueError :", erreur)

Cette `ValueError` est une bonne nouvelle. Sans elle, on alignerait par
erreur deux jeux de donnees de tailles differentes et le resultat serait
silencieusement faux.

Un plantage est une information. Un chiffre faux n'en est pas une.

---

# 2. pandas : charger et inspecter

## 2.1 Le pont avec la seance 3

La liste de dictionnaires de la seance 3 avait deja la forme d'un tableau.
pandas ne fait que lui donner des outils.

In [ ]:
# Seance 3 : une liste de dictionnaires
depenses = [
    {"libelle": "Edf", "categorie": "Logement", "montant": 894.56},
    {"libelle": "Metro", "categorie": "Transport", "montant": 42.10},
]

# Seance 7 : le meme contenu, en DataFrame
pd.DataFrame(depenses)

## 2.2 Le piege du separateur

Le relevé vient d'une banque francaise. En francais, la virgule sert deja
aux decimales : le separateur de colonnes est donc le point-virgule.

On charge d'abord SANS le preciser, pour voir ce que cela donne.

In [ ]:
try:
    mauvais = pd.read_csv(source_releve())
    print("sans sep :", mauvais.shape)
except Exception as erreur:
    print(type(erreur).__name__, ":", erreur)

pandas proteste. Mais lis son message : il dit « j'attendais 1 champ, j'en
ai vu 2 ». Il ne parle **jamais** de separateur.

Pose-toi la question : qu'est-ce que ce message te dit de faire ? Rien.
C'est le vrai enseignement de cette cellule : une erreur peut etre bruyante
et parfaitement muette sur sa cause.

L'erreur tombe ici parce que les montants contiennent des virgules
(`"40,37 EUR"`) : le nombre de champs varie d'une ligne a l'autre, et
pandas s'en apercoit. **Sur un fichier sans aucune virgule, le meme oubli
passerait en silence** : une seule colonne, `(418, 1)`, et aucun message.

D'ou le reflexe : regarder `df.shape` avant meme `head()`.

In [ ]:
df = pd.read_csv(source_releve(), sep=";", encoding="utf-8")
print("avec sep :", df.shape)
df.head()

## 2.3 Le rituel des cinq commandes

Devant tout jeu de donnees inconnu, dans cet ordre. A imprimer et afficher
au mur.

In [ ]:
print(df.shape)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

`describe()` ne trouve **aucune** colonne numerique. C'est normal : les
montants sont encore du texte. C'est la meilleure transition possible vers
le nettoyage.

Note : en pandas 3.0, les colonnes de texte s'affichent avec le type `str`
(backend Arrow) et non plus `object`. Les tutoriels qui parlent d'`object`
sont dates.

In [ ]:
df["categorie"].value_counts()

Vingt orthographes pour cinq categories. C'est `value_counts()` qui revele
que `Logement`, `logement`, `LOGEMENT` et `  Logement ` sont quatre
categories differentes pour la machine.

Une somme par categorie faite maintenant serait fausse, et personne ne s'en
apercevrait.

In [ ]:
# Et pour voir les trous d'un coup :
df.isna().sum()

---

# 3. Selectionner et filtrer

## 3.1 `.loc` et `.iloc`

`.iloc` parle **positions**. `.loc` parle **etiquettes**. Le `i` de `.iloc`
est celui de *index entier*.

In [ ]:
print("iloc[0] - la premiere ligne, quoi qu'il arrive")
print(df.iloc[0])

In [ ]:
# Sur les tranches, la difference surprend tout le monde.
print("iloc[0:3] ->", len(df.iloc[0:3]), "lignes (la fin est EXCLUE)")
print("loc[0:3]  ->", len(df.loc[0:3]), "lignes (la fin est INCLUSE)")

Ce n'est pas une incoherence. `.iloc` suit la convention Python (`range`,
`liste[0:3]`). `.loc` decoupe par etiquettes : si les etiquettes sont des
dates, `df.loc["2026-01":"2026-06"]` doit inclure juin, sinon la syntaxe
serait inutilisable.

Le moment ou cela mord : apres un filtrage, les etiquettes ne sont plus
`0, 1, 2...`. `.iloc[0]` marche toujours, `.loc[0]` peut lever une
`KeyError`.

## 3.2 Combiner deux filtres

`and` attend deux valeurs vrai/faux. Ici on lui donne deux colonnes de
418 booleens : il ne sait pas quoi en faire.

In [ ]:
petit = pd.DataFrame({"montant": [50.0, 150.0, 300.0],
                      "categorie": ["Loisirs", "Loisirs", "Logement"]})

try:
    petit[petit.montant > 100 and petit.categorie == "Loisirs"]
except ValueError as erreur:
    print("ValueError :", erreur)

In [ ]:
# La forme correcte : & au lieu de and, et des parentheses.
petit[(petit.montant > 100) & (petit.categorie == "Loisirs")]

Les parentheses ne sont pas decoratives : en Python, `&` est prioritaire sur
`>`. Sans elles, `petit.montant > 100 & petit.categorie` se lit
`petit.montant > (100 & petit.categorie)`.

Variante plus lisible, ou `and` fonctionne parce que la chaine est analysee
par pandas et non par Python :

In [ ]:
petit.query("montant > 100 and categorie == 'Loisirs'")

---

# 4. Nettoyer

Six defauts ont ete plantes dans ce fichier, et aucun n'est artificiel :
tous se rencontrent sur un vrai export bancaire.

| Le defaut | Ce qu'on en fait |
|---|---|
| trois formats de date, dont des vides | deux formats explicites, puis `fillna` |
| montants en texte | `to_numeric(errors="coerce")` |
| casse et espaces incoherents | `.str.strip().str.title()` |
| 18 doublons exacts | `.drop_duplicates()` |
| 8 montants multiplies par 1000 | un seuil, et on les ecarte |
| 19 remboursements | separes, pas supprimes |

## 4.1 Le piege des dates

C'est le moment le plus important de la seance. Voici ce que proposent la
plupart des tutoriels.

In [ ]:
texte = df["date_operation"].str.strip().str.replace("/", "-", regex=False)

piege = pd.to_datetime(texte, format="mixed", dayfirst=True, errors="coerce")

print("dates converties :", piege.notna().sum(), "sur", len(df))
print("aucune erreur, aucun avertissement")

Tout va bien. Sauf que :

In [ ]:
print("minimum :", piege.min())
print("maximum :", piege.max())

Le releve va de janvier a juin. Pourquoi le maximum est-il en decembre ?

`dayfirst=True` s'applique aussi aux dates deja au format ISO. Quand le jour
et le mois sont tous deux inferieurs ou egaux a 12, pandas les inverse, sans
rien dire.

In [ ]:
for exemple in ["2026-04-12", "2026-06-07", "2026-05-02"]:
    lu = pd.to_datetime(exemple, format="mixed", dayfirst=True)
    print(f"{exemple}  lu comme  {lu.date()}")

### La parade : deux formats explicites

Chaque tentative ne reconnait qu'un seul format et met `NaT` partout
ailleurs. `fillna` recolle les deux. Ce qui reste vide est vraiment
illisible.

In [ ]:
iso = pd.to_datetime(texte, format="%Y-%m-%d", errors="coerce")
fr = pd.to_datetime(texte, format="%d-%m-%Y", errors="coerce")

dates = iso.fillna(fr)

print("dates ISO       :", iso.notna().sum())
print("dates FR        :", (iso.isna() & fr.notna()).sum())
print("illisibles      :", dates.isna().sum())
print()
print("minimum :", dates.min())
print("maximum :", dates.max())

# On compte les degats de la methode precedente.
differentes = (piege != dates) & piege.notna() & dates.notna()
print()
print("dates que la methode 'mixed' avait faussees :", differentes.sum())

In [ ]:
df = df.assign(date_operation=dates)

La regle a retenir : **une conversion qui ne leve pas d'erreur n'est pas une
conversion correcte**. Apres toute conversion de dates, afficher le minimum
et le maximum.

## 4.2 Les montants

Le fichier contient `"40,37 EUR"`, `"1 250,00 EUR"`, `"N/A"` et des cases
vides.

In [ ]:
montants_texte = (
    df["montant"]
    .str.replace("EUR", "", regex=False)
    .str.replace(" ", "", regex=False)
    .str.replace(",", ".", regex=False)
)

df = df.assign(montant=pd.to_numeric(montants_texte, errors="coerce"))

print("montants convertis :", df["montant"].notna().sum())
print("non convertibles   :", df["montant"].isna().sum())

L'ordre des remplacements compte. Il faut retirer l'espace **avant** de
traiter la virgule :

- espace puis virgule : `"1 250,00"` donne `"1250,00"` puis `"1250.00"`
- virgule seule : `"1 250.00"` reste du texte et `to_numeric` echoue

`errors="coerce"` met `NaN` la ou la conversion echoue au lieu de tout
arreter. On decide ensuite quoi faire de ces trous : c'est une decision
d'analyste, pas une panne.

## 4.3 La casse et les espaces

In [ ]:
print("avant :", df["categorie"].nunique(), "valeurs distinctes")

categorie = df["categorie"].fillna("").str.strip().str.title().replace("", "Autre")
df = df.assign(
    categorie=categorie,
    libelle=df["libelle"].str.strip().str.title(),
)

print("apres :", df["categorie"].nunique(), "valeurs distinctes")
df["categorie"].value_counts()

`.str` est la passerelle : elle applique une methode de texte a toute la
colonne, sans ecrire de boucle.

Attention, `.str` propage les `NaN` : `NaN.strip()` n'a pas de sens. C'est
pourquoi on fait `fillna("")` avant, sinon les categories manquantes
resteraient `NaN` au lieu de devenir `Autre`.

## 4.4 Les doublons

In [ ]:
avant = len(df)
df = df.drop_duplicates()
print(f"doublons supprimes : {avant} -> {len(df)}  ({avant - len(df)} lignes)")

## 4.5 `dropna` : la faute couteuse

`dropna()` sans argument supprime toute ligne ayant au moins un trou,
toutes colonnes confondues.

In [ ]:
print("dropna() tout court            :", len(df.dropna()))
print("dropna(subset=[date, montant]) :",
      len(df.dropna(subset=["date_operation", "montant"])))
print()
print("colonne 'note' vide :", df["note"].isna().sum(), "fois sur", len(df))

La colonne `note` est vide sept fois sur dix, et c'est **normal** : on ne
commente pas chaque achat. Sans `subset`, elle emporte les trois quarts du
releve avec elle.

La regle : **on ne jette jamais une ligne pour une colonne dont on n'a pas
besoin.**

Le raisonnement se fait colonne par colonne :

| Colonne | Decision | Pourquoi |
|---|---|---|
| `date_operation` | jeter la ligne | une depense sans date est inexploitable |
| `montant` | jeter la ligne | sans montant, il n'y a rien a analyser |
| `categorie` | boucher avec `Autre` | la depense existe, on ignore son poste |
| `note` | ne rien faire | son absence n'est pas une information manquante |

In [ ]:
df = df.dropna(subset=["date_operation", "montant"])
print("lignes restantes :", len(df))

## 4.6 Les montants aberrants

Une valeur techniquement valide peut etre manifestement fausse : ici huit
montants ont ete multiplies par 1000 par une virgule mal placee a la saisie.

Le seuil ne sort pas des donnees : il sort de la connaissance du domaine.
Un loyer de 900 euros existe ; un achat de 340 000 euros sur un releve
personnel, non.

In [ ]:
SEUIL = 5000.0

avant = len(df)
df = df[df["montant"].abs() < SEUIL]
print(f"montants aberrants retires : {avant} -> {len(df)}")

# Comparer la moyenne et la mediane est un detecteur d'anomalie en une ligne.
print()
print("moyenne :", round(df["montant"].mean(), 2))
print("mediane :", round(df["montant"].median(), 2))

## 4.7 Les negatifs ne sont pas des erreurs

Certaines lignes ont un montant negatif. Ce ne sont pas des fautes de
saisie : ce sont des remboursements. On les **separe**, on ne les supprime
pas.

Jeter une donnee valide est une faute plus grave que garder une donnee
sale : la donnee sale se voit, la donnee jetee non.

In [ ]:
depenses = df[df["montant"] > 0].copy()
remboursements = df[df["montant"] < 0].copy()

print("depenses       :", len(depenses))
print("remboursements :", len(remboursements),
      "pour", round(remboursements["montant"].sum(), 2), "EUR")

In [ ]:
# Deux colonnes calculees, utiles pour la suite.
depenses = depenses.assign(
    mois=depenses["date_operation"].dt.to_period("M").astype(str),
    jour_semaine=depenses["date_operation"].dt.day_name(),
)
depenses.head()

---

# 5. pandas 3.0 : Copy-on-Write

La seule regle a retenir : **pandas rend une nouvelle table. On reaffecte.**

Regardons ce que fait l'affectation dite chainee.

In [ ]:
import warnings

essai = pd.DataFrame({"montant": [50.0, 150.0, 300.0]})

with warnings.catch_warnings(record=True) as messages:
    warnings.simplefilter("always")
    essai[essai.montant > 100]["grosse"] = True
    for message in messages:
        print(type(message.message).__name__, ":", str(message.message)[:70])

print()
print("la colonne 'grosse' existe-t-elle ?", "grosse" in essai.columns)

Point important, souvent mal rapporte : ce n'est **pas** une exception.
pandas emet un `ChainedAssignmentError` comme **avertissement**, ne modifie
rien, et le programme continue.

Un plantage aurait alerte. Ici, l'analyse se poursuit sur des donnees
inchangees : c'est plus sournois.

Pourquoi ? Parce qu'il y a deux operations dans cette ligne. `essai[filtre]`
construit une nouvelle table (l'indexation booleenne copie), puis on ajoute
une colonne a cette table intermediaire, que personne ne garde.

In [ ]:
# La forme correcte : une seule indexation, avec .loc
essai.loc[essai.montant > 100, "grosse"] = True
essai

## Creer une colonne : deux gestes corrects

Les deux fonctionnent en pandas 3.0. Le second est prefere parce qu'il rend
une table, donc il s'enchaine.

In [ ]:
# Correct, mais se relit mal en serie.
# depenses["annee"] = depenses["date_operation"].dt.year

# Prefere : rend une nouvelle table.
depenses = depenses.assign(annee=depenses["date_operation"].dt.year)
depenses[["date_operation", "annee"]].head(3)

A ne pas confondre : `df["x"] = ...` n'est faux que **derriere un filtre**.
Sur une table entiere, c'est parfaitement legitime et c'est meme la forme la
plus lisible pour une colonne isolee.

---

# 6. Agreger

Trois gestes, toujours dans le meme ordre : **separer** en piles,
**calculer** sur chaque pile, **recoller** les resultats.

In [ ]:
depenses.groupby("categorie")["montant"].sum().sort_values(ascending=False)

Six chiffres remplacent plus de trois cents lignes.

A noter : `depenses.groupby("categorie")` seul ne calcule **rien**. Il rend
un objet paresseux. C'est l'agregation qui declenche le travail.

In [ ]:
print(depenses.groupby("categorie"))

## 6.1 Plusieurs calculs d'un coup

La forme `nom=("colonne", "fonction")` nomme les colonnes de sortie. C'est
la seule a retenir : l'autre produit des index a plusieurs niveaux,
illisibles.

In [ ]:
resume = (
    depenses.groupby("categorie")
    .agg(
        nombre=("montant", "count"),
        total=("montant", "sum"),
        moyenne=("montant", "mean"),
        mediane=("montant", "median"),
        maximum=("montant", "max"),
    )
    .sort_values("total", ascending=False)
    .reset_index()
)
resume.round(2)

Lis la ligne `Transport` et la ligne `Logement` a voix haute.

Le transport est le poste le **plus frequent** et le **moins cher**. Le
logement est le moins frequent et de loin le plus lourd.

Une moyenne seule ment. Un `count` pose a cote d'elle dit la verite.

Regarde aussi la ligne `Autre` : sa moyenne est nettement au-dessus de sa
mediane. C'est le signe qu'une valeur atypique la tire vers le haut.

## 6.2 Le tableau croise

C'est le tableau croise dynamique d'Excel, en une ligne, et reproductible.

In [ ]:
croise = depenses.pivot_table(
    index="categorie",
    columns="mois",
    values="montant",
    aggfunc="sum",
    fill_value=0,
    margins=True,
)
croise.round(0)

`fill_value=0` bouche les trous, `margins=True` ajoute la ligne et la
colonne de totaux.

Memes chiffres que la forme longue ci-dessous - mais personne ne repere une
anomalie dans une liste de 36 lignes, alors qu'elle saute aux yeux dans un
tableau croise.

In [ ]:
depenses.groupby(["categorie", "mois"])["montant"].sum().reset_index(name="total").head(8)

## 6.3 Les parts, et le top 5

In [ ]:
totaux = depenses.groupby("categorie")["montant"].sum()
parts = (totaux / totaux.sum() * 100).sort_values(ascending=False)
parts.round(1)

In [ ]:
depenses.nlargest(5, "montant")[["date_operation", "libelle", "categorie", "montant"]]

## 6.4 Exporter

Trois decisions dans une seule ligne.

In [ ]:
depenses.to_csv("releve_propre.csv", index=False, encoding="utf-8")
print("ecrit : releve_propre.csv", "-", len(depenses), "lignes")

- `index=False` : sinon pandas ecrit une premiere colonne sans nom
  contenant `0, 1, 2...`, qui devient `Unnamed: 0` a la relecture. C'est
  l'oubli le plus frequent.
- `encoding="utf-8"` : pour que les accents survivent au voyage.
- `sep=";"` serait a ajouter si le destinataire ouvre le fichier dans Excel
  en francais.

Les memes conventions qui t'ont piege a la lecture piegent ton destinataire
a l'ecriture.

---

# 7. A toi

Cinq questions. Chacune se repond par une ou deux expressions pandas.
Ecris ta reponse dans la cellule qui suit, puis verifie le resultat.

**Q1.** Quel est le mois le plus lourd, et de combien depasse-t-il le mois
le plus leger ?

In [ ]:
# Ta reponse ici

**Q2.** Combien de depenses par moyen de paiement ?

In [ ]:
# Ta reponse ici

**Q3.** Quel jour de la semaine depense-t-on le plus, en total ? Et en
moyenne ? Les deux reponses sont-elles les memes ?

In [ ]:
# Ta reponse ici

**Q4.** Quelle categorie a la plus grosse depense unique ? Est-ce aussi
celle qui coute le plus cher au total ?

In [ ]:
# Ta reponse ici

**Q5.** Recharge le fichier brut et refais tout le nettoyage en une seule
expression chainee, avec `.pipe()`. Affiche le nombre de lignes apres
chaque etape.

In [ ]:
# Ta reponse ici

---

## Ce qu'il faut retenir

1. **La vitesse vient du rangement**, pas de la magie.
2. **pandas rend une nouvelle table. On reaffecte.**
3. **Un plantage est une information ; un chiffre faux n'en est pas une.**
4. **Une conversion qui ne leve pas d'erreur n'est pas une conversion
   correcte.** Apres des dates, afficher le minimum et le maximum.
5. **On ne jette jamais une ligne pour une colonne dont on n'a pas besoin.**
6. **On trace ce qu'on jette.** Un nettoyage silencieux est un nettoyage
   suspect.

## Les corriges

Les corriges du TD sont dans `seances/s07-numpy-pandas/corrige/reprise/`.
Le corrige de reference du projet est dans `fil-rouge/v4-donnees/`.

## Bloque ?

Colle le message d'erreur **complet, en texte** - jamais une capture - dans
le flux du cours sur Google Classroom. La derniere ligne d'un traceback dit
toujours ce qui ne va pas.